### How to Enable a GPU in Google Colab

To enable a GPU in this notebook, please follow these steps:
1. In the top menu, navigate to **Runtime** > **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU** (or any other available GPU option).
3. Click **Save**.

Once completed, run the cells below to verify that the GPU is available and ready for use.

In [ ]:
import torch
import tensorflow as tf

# 1. Verify GPU availability with PyTorch
print("--- PyTorch GPU Verification ---")
is_torch_cuda = torch.cuda.is_available()
print(f"Is GPU available in PyTorch?: {is_torch_cuda}")
if is_torch_cuda:
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

print("\n--- TensorFlow GPU Verification ---")
# 2. Verify GPU availability with TensorFlow
gpus = tf.config.list_physical_devices('GPU')
is_tf_gpu = len(gpus) > 0
print(f"Is GPU available in TensorFlow?: {is_tf_gpu}")
if is_tf_gpu:
    for gpu in gpus:
        print(f"GPU Device: {gpu}")


In [ ]:
import sys
import torch

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA version used by PyTorch:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%pip install ultralytics

In [ ]:
import ultralytics
import torch

print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

In [ ]:
from ultralytics import YOLO
import torch

model = YOLO("yolo11n.pt")

print("YOLO model loaded successfully")
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import zipfile
import os

zip_path = "/content/football-players-detection.v15-yolo11n.yolov11.zip"
extract_dir = "/content/football_dataset"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Dataset extracted successfully!")
print("\nFiles and folders:")

for item in os.listdir(extract_dir):
    print("-", item)

In [ ]:
import os
import yaml

dataset_dir = "/content/football_dataset"
yaml_path = os.path.join(dataset_dir, "data.yaml")

# Read the dataset configuration
with open(yaml_path, "r") as file:
    data = yaml.safe_load(file)

print("=== Dataset Configuration ===")
print("Classes:", data.get("names"))
print("Number of classes:", data.get("nc"))

print("\n=== Dataset Folders ===")
for split in ["train", "valid", "test"]:
    images_dir = os.path.join(dataset_dir, split, "images")
    labels_dir = os.path.join(dataset_dir, split, "labels")

    image_count = len(os.listdir(images_dir)) if os.path.isdir(images_dir) else 0
    label_count = len(os.listdir(labels_dir)) if os.path.isdir(labels_dir) else 0

    print(f"{split}: {image_count} images, {label_count} label files")

print("\n=== YAML Paths ===")
for key in ["train", "val", "test"]:
    print(f"{key}: {data.get(key)}")

In [ ]:
from ultralytics.data.utils import check_det_dataset

data_yaml = "/content/football_dataset/data.yaml"

dataset_info = check_det_dataset(data_yaml)

print("Dataset configuration loaded successfully!")
print("Number of classes:", dataset_info["nc"])
print("Class names:", dataset_info["names"])
print("Training images directory:", dataset_info["train"])
print("Validation images directory:", dataset_info["val"])
print("Test images directory:", dataset_info.get("test"))

In [ ]:
from ultralytics import YOLO
import torch

# 1. Verify GPU availability
assert torch.cuda.is_available(), "GPU not available! Check Colab Runtime settings."

print("GPU:", torch.cuda.get_device_name(0))

# 2. Load pretrained YOLO11n weights
model = YOLO("yolo11n.pt")

# 3. Train on your football dataset
results = model.train(
    data="/content/football_dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    seed=42,
    project="/content/football_training",
    name="yolo11n_football",
    plots=True
)

In [ ]:
from ultralytics import YOLO
from IPython.display import display, Image
import glob
import os

# Load the best trained checkpoint
model = YOLO(
    "/content/football_training/yolo11n_football/weights/best.pt"
)

# Select one image from the held-out test set
test_images = glob.glob("/content/football_dataset/test/images/*")
test_images = [
    p for p in test_images
    if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
]

print("Test images available:", len(test_images))

# Run prediction on the first test image
image_path = test_images[0]
print("Testing image:", os.path.basename(image_path))

results = model.predict(
    source=image_path,
    imgsz=640,
    conf=0.25,
    save=True
)

# Display the image with predicted bounding boxes
output_path = results[0].save_dir
predicted_image = os.path.join(
    output_path, os.path.basename(image_path)
)

display(Image(filename=predicted_image))

In [ ]:
from IPython.display import display, Image

display(
    Image(filename="/content/runs/detect/predict/798b45_3_3_png.rf.95da6d150b7974069511bdff14c48f0f.jpg")
)

In [ ]:
from ultralytics import YOLO

# Load your best trained model
model = YOLO(
    "/content/football_training/yolo11n_football/weights/best.pt"
)

# Evaluate on the held-out test dataset
results = model.val(
    data="/content/football_dataset/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project="/content/football_training",
    name="test_evaluation"
)

print("Test evaluation completed!")

In [ ]:
from ultralytics import YOLO
import os

# Load your best trained football model
model_path = "/content/football_training/yolo11n_football/weights/best.pt"
model = YOLO(model_path)

# Export to ONNX format
onnx_path = model.export(format="onnx", imgsz=640)

print("ONNX export completed!")
print("Exported model:", onnx_path)
print("File exists:", os.path.exists(onnx_path))

In [ ]:
import os

weights_dir = "/content/football_training/yolo11n_football/weights"

for filename in ["best.pt", "best.onnx"]:
    path = os.path.join(weights_dir, filename)

    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"✅ {filename}: {size_mb:.2f} MB")
    else:
        print(f"❌ {filename}: NOT FOUND")

In [ ]:
from google.colab import files
import os
import zipfile

weights_dir = "/content/football_training/yolo11n_football/weights"
zip_path = "/content/football_yolo11n_models.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(
        os.path.join(weights_dir, "best.pt"),
        arcname="best.pt"
    )
    zipf.write(
        os.path.join(weights_dir, "best.onnx"),
        arcname="best.onnx"
    )

print("Archive created successfully!")
files.download(zip_path)

In [ ]:
from ultralytics import YOLO
import glob
import os

# Load the exported ONNX model
onnx_path = (
    "/content/football_training/yolo11n_football/"
    "weights/best.onnx"
)
model = YOLO(onnx_path, task="detect")

# Select an image from the test set
test_images = glob.glob("/content/football_dataset/test/images/*")
test_images = [
    p for p in test_images
    if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
]

image_path = test_images[0]

# Run inference using ONNX
results = model.predict(
    source=image_path,
    imgsz=640,
    conf=0.25,
    save=True
)

print("ONNX inference successful!")
print("Image:", os.path.basename(image_path))
print("Detections:", len(results[0].boxes))

for class_id, count in results[0].names.items():
    detected = sum(
        int(box.cls.item()) == class_id
        for box in results[0].boxes
    )
    print(f"{count}: {detected}")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

model_path = "/content/football_training/yolo11n_football/weights/best.pt"

if os.path.exists(model_path):
    print("✅ Trained model found!")
    print(f"Model size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("❌ Trained model not found. We need to upload it again.")

In [ ]:
from ultralytics import YOLO
import torch

model_path = "/content/football_training/yolo11n_football/weights/best.pt"
video_path = "/content/videoplayback (4).mp4"

# Load your trained model
model = YOLO(model_path)

# Check GPU availability
device = 0 if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if device == 0 else "CPU")

# Run object detection on the video
results = model.predict(
    source=video_path,
    imgsz=640,
    conf=0.25,
    device=device,
    save=True,
    project="/content/football_video",
    name="prediction",
    exist_ok=True,
    verbose=False
)

print("✅ Video processing finished!")
print("Output folder: /content/football_video/prediction")

In [ ]:
import os

output_dir = "/content/football_video/prediction"

for filename in os.listdir(output_dir):
    filepath = os.path.join(output_dir, filename)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)

    print(f"File: {filename}")
    print(f"Size: {size_mb:.2f} MB")
    print(f"Path: {filepath}")

In [ ]:
import subprocess
import os

input_video = "/content/football_video/prediction/videoplayback (4).avi"
output_video = "/content/football_video/prediction/football_detection.mp4"

command = [
    "ffmpeg", "-y",
    "-i", input_video,
    "-c:v", "libx264",
    "-preset", "fast",
    "-crf", "23",
    "-pix_fmt", "yuv420p",
    "-movflags", "+faststart",
    "-c:a", "aac",
    "-b:a", "128k",
    output_video
]

result = subprocess.run(command, capture_output=True, text=True)

if result.returncode == 0 and os.path.exists(output_video):
    size_mb = os.path.getsize(output_video) / (1024 * 1024)
    print("✅ MP4 conversion successful!")
    print(f"File: {output_video}")
    print(f"Size: {size_mb:.2f} MB")
else:
    print("❌ Conversion failed:")
    print(result.stderr[-3000:])